# unit02 レッスン: 検証設計とリーク

**題材** — 複数のECサイトから収集した商品リストから、**実売価格 `price` を当てる**回帰コンペ。
評価指標は unit01 と同じ **RMSLE**。データは `data/` に同梱済みで、ネットワークは一切使わない。

unit01 との決定的な違いが2つある。

1. **`train` は 2025-10-01〜2026-02-28、`test` は 2026-03-01 以降。** つまりこのコンペは「**未来を当てる**」
2. **同じ商品が複数のサイトから重複して収集されている。** 1商品あたり 1〜4 行ある

この2つの構造が、これから見る「**手元のスコアが平気で嘘をつく**」現象の正体になる。

## このレッスンを終えると作れるようになるもの

1. KFold で **OOF(out-of-fold)予測**を自分の手で組める。全行に「その行を学習に使っていないモデルによる予測」が1つずつ入った配列を作れる
2. **グループリーク**を見つけて `GroupKFold` で潰せる。**モデルもデータも一切変えず、分割方法だけで検証スコアが 0.61 → 1.11 と倍近く動く**のを実測する
3. **時間リーク**を `TimeSeriesSplit` で潰せる。ついでに TimeSeriesSplit 特有の落とし穴(**全行は検証されない**)を実際に踏んで直せる
4. **ターゲットリーク**を3つの方法で検出できる。「スコアが良すぎるときは、喜ぶ前に疑う」

所要の目安: **90〜120分**。このあと演習 `ex01`〜`ex04` が続く。

## 今日の一行

> **手元のスコアが信じられないなら、改善サイクルは回らない。**
> Kaggle で最も差がつくのはモデルではなく検証設計だ、と言われるのはこのため。

## このレッスンの読み方

セルは**上から順に**実行する。構成は unit01 と同じ8ステップの繰り返し:

| 記号 | 内容 |
|---|---|
| ① | なぜこれを学ぶのか(実務のどこで使うか) |
| ② | 解説(C# との対応表・API 表) |
| ③ | **見る** — 完成コードを実行して結果を見る |
| ④ | **予測する** — 次のセルの結果を頭の中で予測する |
| ⑤ | **変えてみる** — 実行して予測と照合する |
| ⑥ | **書いてみる**(指示) |
| ⑦ | 自分で書くセル(`# ここに書く`) |
| ⑧ | チェックポイント(即時採点) |

⑦ を書かずに実行しても notebook は止まらない。⑧ が `[NG]` を出して、何が期待値なのかを教えてくれる。

In [2]:
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    KFold,             # ランダムにK分割する
    StratifiedKFold,   # クラス比率を保ってK分割する(分類向け)
    GroupKFold,        # 同じグループが train と valid に跨らないようにK分割する
    TimeSeriesSplit,   # 時間順に「過去で学習 → 未来で検証」を繰り返す
    train_test_split,  # 1回だけの分割(ホールドアウト)
)
from lightgbm import LGBMRegressor   # 勾配ブースティング木。詳細は unit03。今日は「検証の道具」として最小限だけ使う

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# データの場所。notebook をユニット直下で開いても、リポジトリのルートで開いても動くようにする。
DATA = Path("data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit02-validation-and-leakage/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

train = pd.read_csv(DATA / "train.csv", parse_dates=["collected_at"])
test = pd.read_csv(DATA / "test.csv", parse_dates=["collected_at"])
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

print("train:", train.shape, " test:", test.shape, " sample_submission:", sample_submission.shape)
print("train 期間:", train["collected_at"].min().date(), "〜", train["collected_at"].max().date())
print("test  期間:", test["collected_at"].min().date(), "〜", test["collected_at"].max().date(), " ← 未来")
print("train にしかない列:", sorted(set(train.columns) - set(test.columns)))

# ---------- 目的変数は「対数空間」で扱う ----------
# 評価指標が RMSLE なので、log1p に移してしまえば「ただの RMSE」になる(unit01 概念1でやった話)。
#   RMSLE(y, p) = RMSE(log1p(y), log1p(p))
# 以後 y はずっと「log1p した価格」を指す。予測も対数空間で出す。
y = np.log1p(train["price"].to_numpy(dtype=float))
print("\ny =", y.shape, "  log1p した価格。平均", round(float(y.mean()), 4))


def rmse_log(a, b):
    """すでに log1p 済みの2つの配列の RMSE を返す。= 元の価格での RMSLE。"""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.sqrt(np.mean((a - b) ** 2)))


# ---------- 本番LB(答え合わせ用) ----------
# コンペ中は当然見えない。教材なので「本当の答え」を持っていて、いつでも本番スコアを出せる。
# このレッスンでは「手元のCVが本番をどれだけ言い当てたか」を毎回突き合わせるために使う。
_UNIT_DIR = DATA.resolve().parent
ANSWER = _UNIT_DIR.parent / ".solutions" / _UNIT_DIR.name / "_answer.csv"
HAS_ANSWER = ANSWER.exists()
if HAS_ANSWER:
    y_test = np.log1p(
        pd.read_csv(ANSWER).set_index("record_id").loc[test["record_id"], "price"].to_numpy(dtype=float)
    )


def lb_score(pred_log):
    """本番LB(未来の test)での RMSLE。答えファイルが無ければ nan を返す。"""
    if not HAS_ANSWER:
        return float("nan")
    return rmse_log(pred_log, y_test)


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def check_frame(name, actual, shape=None, columns=None, hint=""):
    """DataFrame の形と列名を採点する。DataFrame でなくても例外にしない。"""
    if not isinstance(actual, pd.DataFrame):
        print(f"[NG] {name}: 期待値 pandas.DataFrame(shape={shape}, columns={columns}) / 実際 {type(actual).__name__}")
        if hint:
            print(f"     ヒント: {hint}")
        return False
    problems = []
    if shape is not None and tuple(actual.shape) != tuple(shape):
        problems.append(f"shape の期待値 {tuple(shape)} / 実際 {tuple(actual.shape)}")
    if columns is not None and list(actual.columns) != list(columns):
        problems.append(f"列名の期待値 {list(columns)} / 実際 {list(actual.columns)}")
    if problems:
        print(f"[NG] {name}: " + " | ".join(problems))
        if hint:
            print(f"     ヒント: {hint}")
        return False
    print(f"[OK] {name}: 正解!")
    return True


def call_safely(fn, *args, **kwargs):
    """未完成の関数を呼んでも notebook が止まらないようにするラッパ。例外なら None を返す。"""
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数の中で例外が出ました → {type(e).__name__}: {e})")
        return None


def arr_stat(a, how):
    """配列の統計を安全に取り出す。取れなければ None(未記入でも止まらないため)。"""
    try:
        a = np.asarray(a, dtype=float)
    except Exception:
        return None
    if a.ndim != 1 or a.size == 0:
        return None
    table = {"len": lambda: float(a.size), "nan": lambda: float(np.isnan(a).sum()),
             "mean": lambda: float(np.nanmean(a)), "min": lambda: float(np.nanmin(a)),
             "max": lambda: float(np.nanmax(a)), "nunique": lambda: float(len(np.unique(np.round(a, 9))))}
    try:
        return table[how]()
    except Exception:
        return None


print("\nセットアップ完了。ヘルパー: rmse_log / lb_score / check / check_frame / call_safely / arr_stat")

train: (1661, 13)  test: (330, 11)  sample_submission: (330, 2)
train 期間: 2025-10-01 〜 2026-02-28
test  期間: 2026-03-01 〜 2026-03-29  ← 未来
train にしかない列: ['discount_rate', 'price']

y = (1661,)   log1p した価格。平均 9.0283

セットアップ完了。ヘルパー: rmse_log / lb_score / check / check_frame / call_safely / arr_stat


---
# 概念1 — 手元のスコアを信じられるようにする(CV と OOF)

## ① なぜ: 1回の分割で出したスコアは、運で 0.05 動く

コンペでも実務でも、1日の仕事はこの繰り返しになる。

> 「新しい特徴量を足してみた。スコアが 0.005 良くなった。**採用していい?**」

この問いに答えるには、**手元の検証スコアがどれくらい揺れるのか**を知っていないといけない。
揺れ幅が 0.05 あるなら、0.005 の改善は「良くなった」ではなく「たまたま」だ。
それを採用し続けると、**手元では上がり続けるのに本番では下がる**という、コンペで最もよくある事故が起きる。

実務でも同じ構図だ。「モデルAとモデルB、どちらを本番に載せるか」を1回のホールドアウトで決めると、
分割の乱数が違うだけで結論がひっくり返る。だから**まず検証の作り方を固める**。今日の主題はそこ、モデルではない。

## ② 解説: ホールドアウト → K分割交差検証 → OOF 予測

### ホールドアウト(unit01 の ex03 でやったやつ)

データを1回だけ「学習用」と「検証用」に割る。速いが、**割り方の運**がそのままスコアに乗る。

```
[■■■■■■■■□□]   ■=学習(80%)  □=検証(20%)  ← 割り方は乱数次第
```

### K分割交差検証(K-Fold Cross Validation)

データをK個のブロックに割り、**「1つを検証、残りを学習」をK回**繰り返す。

```
fold 0:  [□□][■■][■■][■■][■■]      ■=学習  □=検証
fold 1:  [■■][□□][■■][■■][■■]
fold 2:  [■■][■■][□□][■■][■■]
fold 3:  [■■][■■][■■][□□][■■]
fold 4:  [■■][■■][■■][■■][□□]
          ↑ 縦に見ると、どの行もちょうど1回だけ □(検証)になっている
```

### OOF 予測(out-of-fold prediction) — 今日いちばん大事な道具

上の図を**縦に**見ると、各行はちょうど1回だけ検証側にいる。そのとき出た予測値を、
**元の行の位置に詰め戻して**1本の配列にしたものが **OOF 予測**だ。

> **OOF 予測 = 全行に「その行を学習に使っていないモデルによる予測」が1つずつ入った配列**

長さは `len(train)` と同じ。これがあると、**train 全体に対して1つのスコア**が出せる。
しかも「答えを見ていない予測」なので、本番に近い。unit10 のアンサンブルでも、この配列が主役になる。

**C# で書くとこうなる:**

```csharp
var oof = new double[rows.Count];                     // 空の箱を用意
foreach (var (trainIdx, validIdx) in splitter.Split(rows))   // K回まわる
{
    var model = new Model();                          // 毎回まっさらな別インスタンス
    model.Fit(rows.Where((_, i) => trainIdx.Contains(i)));
    var pred = model.Predict(rows.Where((_, i) => validIdx.Contains(i)));
    foreach (var (i, p) in validIdx.Zip(pred)) oof[i] = p;   // 元の位置に詰め戻す
}
```

ポイントは3つ。**(a) 箱を先に作る**、**(b) fold ごとにモデルを作り直す**(前の fold の学習結果を持ち越さない)、
**(c) 元の位置に詰め戻す**。Python でもそのまま同じ構造を書く。

### `splitter.split()` は「インデックスの組を yield するイテレータ」

sklearn の splitter は、**データそのものを分けて返さない**。返すのは**行番号の配列の組**だけだ。

```python
for tr_idx, va_idx in kf.split(X):
    ...
```

C# の型で書くなら **`IEnumerable<(int[] trainIdx, int[] validIdx)>` を返すメソッド**、
実装は `yield return` の繰り返し。だから:

- `tr_idx` / `va_idx` は **int の NumPy 配列**(位置インデックス)
- 実データを取り出すのは自分の仕事 → `X.iloc[tr_idx]`(DataFrame)、`y[tr_idx]`(NumPy 配列)
- `for` が回る回数が fold 数

`X.iloc[...]` は**位置での行取り**(`X[...]` のラベル取りと違う)。splitter が返すのは位置なので `.iloc` を使う。

### 今日使う splitter の一覧

| splitter | 何を保証するか | 使うべき場面 | これを使わないと起きるリーク |
|---|---|---|---|
| `KFold(n_splits, shuffle, random_state)` | 何も保証しない(ランダム) | 行が互いに独立なとき | — |
| `StratifiedKFold` | 各 fold の**クラス比率**が元と同じ | 分類、特に不均衡データ | 少数クラスが0件の fold ができる |
| `GroupKFold(n_splits)` | 同じ**グループ**が train と valid に**跨らない** | 同じ実体の行が複数ある | **グループリーク**(概念2) |
| `TimeSeriesSplit(n_splits)` | valid が train より**必ず未来** | 未来を予測する課題 | **時間リーク**(概念3) |

> **回帰では `StratifiedKFold` はそのままでは使えない。** 層化(stratify)は「クラスラベル」が前提で、
> 連続値の `price` を渡すとエラーになる。回帰で層化したいときは**目的変数をビンに切って**そのビンで層化する
> (`pd.qcut(y, 10, labels=False)`)。今日は深入りしないが、名前だけ覚えておく。

### 今日使う API

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| 空の OOF 配列 | `np.full(n, np.nan)` | 長さ n の配列(全部 NaN) | **`np.zeros` ではなく NaN で初期化する**(概念3で理由が分かる) |
| 位置で行を取る | `X.iloc[idx]` | DataFrame | `X[idx]` はラベル取り。splitter が返すのは位置 |
| 位置に詰め戻す | `oof[va_idx] = pred` | — | 既習の NumPy のファンシーインデックス代入そのもの |
| 埋まった行 | `~np.isnan(oof)` | ブール配列 | 既習の numpy-masking |

In [3]:
# GOAL: 1回のホールドアウトは運で動くこと、K分割の OOF はそれを均すことを、同じモデルで比べて見る

# STEP 1: 今日使う「いちばん素朴なモデル」を関数にする。
#   やることは1つだけ — 「学習用の行だけを使って key ごとの平均 log price を計算し、検証用の行に貼る」。
#   groupby は C# の GroupBy と同じ「キーでまとめる」操作(本格的には unit04 でやる)。
def fit_predict_mean(key, tr_idx, va_idx):
    """train[key] ごとの平均 log price を tr_idx の行だけから作り、va_idx の行に貼る。

    戻り値: (予測値の配列, 対応表に無かった行数)
    """
    keys = train[key].to_numpy()
    table = pd.Series(y[tr_idx]).groupby(keys[tr_idx]).mean()  # 学習fold だけで作った「キー -> 平均」
    fallback = float(y[tr_idx].mean())                         # 表に無いキー用の退避先(unit01 と同じ発想)
    pred = pd.Series(keys[va_idx]).map(table)                  # 検証fold のキーを表で引く
    n_unknown = int(pred.isna().sum())                         # 引けなかった = そのキーは学習fold に無かった
    return pred.fillna(fallback).to_numpy(dtype=float), n_unknown


# STEP 2: ホールドアウト(1回だけの分割)を、乱数の種だけ変えて8回やってみる
print("--- ホールドアウト(test_size=0.2)を random_state だけ変えて8回 ---")
holdout_scores = []
for rs in range(8):
    tr_idx, va_idx = train_test_split(np.arange(len(train)), test_size=0.2, random_state=rs)
    pred, _ = fit_predict_mean("category", tr_idx, va_idx)
    s = rmse_log(pred, y[va_idx])
    holdout_scores.append(s)
    print(f"  random_state={rs}: train={len(tr_idx)}行 valid={len(va_idx)}行  RMSLE={s:.4f}")
print(f"\n  最小 {min(holdout_scores):.4f} / 最大 {max(holdout_scores):.4f} / 幅 {max(holdout_scores)-min(holdout_scores):.4f}")
print("  → データもモデルも1文字も変えていないのに、この幅が出る。")
print("    『特徴量を足して 0.005 良くなった』を、この上で判断できるだろうか?")

# STEP 3: 同じモデルで OOF 予測を作る。まず空の箱、次にループ、最後に詰め戻し。
print("\n--- KFold(5) で OOF 予測を作る ---")
kf = KFold(n_splits=5, shuffle=True, random_state=0)
oof = np.full(len(train), np.nan)         # (a) 箱を先に作る。NaN = 「まだ誰も予測していない」印
fill_count = np.zeros(len(train), dtype=int)   # 各行が何回書き込まれたかを数える(検算用)

for fold, (tr_idx, va_idx) in enumerate(kf.split(train)):
    # kf.split(...) が返すのは「行番号の配列の組」。データそのものではない。
    pred, n_unknown = fit_predict_mean("category", tr_idx, va_idx)   # (b) fold ごとに作り直す
    oof[va_idx] = pred                                              # (c) 元の位置に詰め戻す
    fill_count[va_idx] += 1
    print(f"  fold {fold}: tr_idx {tr_idx.shape} va_idx {va_idx.shape} "
          f"型={tr_idx.dtype}  この fold だけのスコア={rmse_log(pred, y[va_idx]):.4f}")

# STEP 4: shape を必ず確認する。OOF は「元データと同じ長さ」で「全行ちょうど1回」埋まっているはず。
print("\n--- OOF 配列の検算(shape を print する規律)---")
print("  train の行数      :", len(train))
print("  oof.shape         :", oof.shape, "  ← 元データと同じ長さ")
print("  NaN が残った行数  :", int(np.isnan(oof).sum()), "  ← 0 なら全行が予測された")
print("  fill_count の最小/最大:", fill_count.min(), "/", fill_count.max(), "  ← どちらも 1 なら『ちょうど1回ずつ』")
print("\n  OOF スコア(train 全行) :", round(rmse_log(oof, y), 4))
print("  ホールドアウト8回の平均 :", round(float(np.mean(holdout_scores)), 4))
print("  → 同じくらいの値。だが OOF は『全行を使って1つ』なので、乱数で 0.05 も動かない(次のセルで確認)")

--- ホールドアウト(test_size=0.2)を random_state だけ変えて8回 ---
  random_state=0: train=1328行 valid=333行  RMSLE=0.7461
  random_state=1: train=1328行 valid=333行  RMSLE=0.7641
  random_state=2: train=1328行 valid=333行  RMSLE=0.7552
  random_state=3: train=1328行 valid=333行  RMSLE=0.7512
  random_state=4: train=1328行 valid=333行  RMSLE=0.7527
  random_state=5: train=1328行 valid=333行  RMSLE=0.7630
  random_state=6: train=1328行 valid=333行  RMSLE=0.7684
  random_state=7: train=1328行 valid=333行  RMSLE=0.7148

  最小 0.7148 / 最大 0.7684 / 幅 0.0537
  → データもモデルも1文字も変えていないのに、この幅が出る。
    『特徴量を足して 0.005 良くなった』を、この上で判断できるだろうか?

--- KFold(5) で OOF 予測を作る ---
  fold 0: tr_idx (1328,) va_idx (333,) 型=int64  この fold だけのスコア=0.7461
  fold 1: tr_idx (1329,) va_idx (332,) 型=int64  この fold だけのスコア=0.7497
  fold 2: tr_idx (1329,) va_idx (332,) 型=int64  この fold だけのスコア=0.7248
  fold 3: tr_idx (1329,) va_idx (332,) 型=int64  この fold だけのスコア=0.7894
  fold 4: tr_idx (1329,) va_idx (332,) 型=int64  この fold だけのスコア=0.7168

--- OOF 配列の検算(s

## ④ 予測: fold 数を変えると OOF スコアはどれくらい動く?

③ では `KFold(n_splits=5, shuffle=True, random_state=0)` を使った。次のセルでは分割の設定だけを変える。
実行する前に予測しよう。

1. **`n_splits` を 2 や 10 に変えたら**、OOF スコアはどれくらい動く?
   ホールドアウトで見た **0.05** くらい動く? それとももっと小さい?
2. `n_splits=2` のとき、各 fold の**学習に使える行数**はいくつになる?
   学習データが半分になるのだから、スコアは悪くなるはずでは?
3. **`shuffle=False`**(既定値)にすると何が変わる? このデータは `record_id` 順に並んでいる。
4. どの設定でも、**OOF 配列の NaN の数**は 0 のままだと思う?

> ヒント: `shuffle=False` の KFold は「先頭から順に5等分」する。
> もしデータが日付順や商品順に並んでいたら、何が起きるだろうか?(今日の後半への伏線)

In [4]:
# GOAL: OOF スコアは分割設定にほとんど左右されない = 「1回のホールドアウトより信じられる」ことを確認する


# STEP 1: ③ のループを関数にまとめる。以後ずっとこれを使う。
def run_cv(key, splitter, groups=None):
    """key ごとの平均モデルを splitter で交差検証し、(スコア, 未知キー率, OOF配列) を返す。"""
    oof = np.full(len(train), np.nan)
    n_unknown_total = 0
    n_valid_total = 0
    for tr_idx, va_idx in splitter.split(train, y, groups):
        # split の第3引数が groups。KFold は無視し、GroupKFold だけが使う(インターフェースは共通)
        pred, n_unknown = fit_predict_mean(key, tr_idx, va_idx)
        oof[va_idx] = pred
        n_unknown_total += n_unknown
        n_valid_total += len(va_idx)
    mask = ~np.isnan(oof)                      # 埋まった行だけで採点する(理由は概念3)
    return rmse_log(oof[mask], y[mask]), n_unknown_total / n_valid_total, oof


# STEP 2: 分割の設定だけを変えて比べる
settings = [
    ("KFold(2,  shuffle=True )", KFold(n_splits=2, shuffle=True, random_state=0)),
    ("KFold(5,  shuffle=True )", KFold(n_splits=5, shuffle=True, random_state=0)),
    ("KFold(10, shuffle=True )", KFold(n_splits=10, shuffle=True, random_state=0)),
    ("KFold(5,  shuffle=False)", KFold(n_splits=5, shuffle=False)),
]
print(f"{'分割設定':<26}{'1foldの学習行数':>16}{'OOFスコア':>12}{'NaN':>6}")
print("-" * 62)
for name, splitter in settings:
    score, _, oof_i = run_cv("category", splitter)
    first_tr, _ = next(iter(splitter.split(train)))
    print(f"{name:<24}{len(first_tr):>14}{score:>12.4f}{int(np.isnan(oof_i).sum()):>7}")

print(f"\nホールドアウト8回の幅: {max(holdout_scores) - min(holdout_scores):.4f}")
print("K を 2 から 10 まで振っても OOF スコアの幅は 0.001 程度。")
print("→ 『全行を1回ずつ検証に使う』だけで、判断の土台がここまで安定する。")
print("→ だから Kaggle では『CVスコア』と言えば普通この OOF スコアを指す。")

# STEP 3: shuffle=False の落とし穴を確認しておく
print("\n--- shuffle=False は何をしているか ---")
_, va0 = next(iter(KFold(n_splits=5, shuffle=False).split(train)))
print("  fold 0 の検証行番号(先頭5件):", va0[:5], "... 最後:", va0[-1])
print("  → 先頭から順に切っているだけ。今回は record_id 順(ランダム)なので害はない。")
print("    だが『日付順に並んだデータ』だと、fold 0 の検証は必ず最古の期間になる。")
print("    データの並び順に意味があるかどうかで挙動が変わる、と覚えておく。")

分割設定                            1foldの学習行数      OOFスコア   NaN
--------------------------------------------------------------
KFold(2,  shuffle=True )           830      0.7463      0
KFold(5,  shuffle=True )          1328      0.7458      0
KFold(10, shuffle=True )          1494      0.7462      0
KFold(5,  shuffle=False)          1328      0.7470      0

ホールドアウト8回の幅: 0.0537
K を 2 から 10 まで振っても OOF スコアの幅は 0.001 程度。
→ 『全行を1回ずつ検証に使う』だけで、判断の土台がここまで安定する。
→ だから Kaggle では『CVスコア』と言えば普通この OOF スコアを指す。

--- shuffle=False は何をしているか ---
  fold 0 の検証行番号(先頭5件): [0 1 2 3 4] ... 最後: 332
  → 先頭から順に切っているだけ。今回は record_id 順(ランダム)なので害はない。
    だが『日付順に並んだデータ』だと、fold 0 の検証は必ず最古の期間になる。
    データの並び順に意味があるかどうかで挙動が変わる、と覚えておく。


## ⑥ 書いてみる: OOF 予測を自分の手で組む

OOF は今日以降ずっと使う道具なので、**ループを自分の指で1回書いておく**価値がある。

次のセルで、`condition`(商品の状態)ごとの平均 log price モデルの OOF 予測を作ろう。

- splitter は **`KFold(n_splits=5, shuffle=True, random_state=0)`**(③ と同じ設定)
- `oof_cond` — 長さ `len(train)` の NumPy 配列。**`np.full(len(train), np.nan)` で作り始める**
- `score_cond` — `rmse_log(oof_cond, y)` の値(**Python の float**)

使う道具は全部そろっている:

| やること | 書き方 |
|---|---|
| 箱を作る | `np.full(len(train), np.nan)` |
| fold を回す | `for tr_idx, va_idx in kf.split(train):` |
| 予測する | `pred, n_unknown = fit_predict_mean("condition", tr_idx, va_idx)` |
| 詰め戻す | `oof_cond[va_idx] = pred` |
| 採点する | `rmse_log(oof_cond, y)` |

5〜7行で書ける。書けたら、**NaN が1つも残っていないこと**を自分で確かめる癖をつけよう
(残っていたら、どこかの行が一度も検証側に入っていない)。

In [5]:
oof_cond = None
score_cond = None
# ここに書く(ヒント: 箱を np.full(len(train), np.nan) で作り、for tr_idx, va_idx in KFold(...).split(train): で回して
#           oof_cond[va_idx] = pred と詰め戻す。最後に rmse_log(oof_cond, y) をfloat()で包んで score_cond に入れる)
oof_cond = np.full(len(train), np.nan)
for tr_idx, va_idx in KFold(n_splits=5, shuffle=True, random_state=0).split(train):
    pred, n_unknown = fit_predict_mean("condition", tr_idx, va_idx)
    oof_cond[va_idx] = pred
score_cond = float(rmse_log(oof_cond,y))

print("oof_cond:", None if oof_cond is None else np.asarray(oof_cond).shape,
      " NaN:", None if oof_cond is None else int(np.isnan(np.asarray(oof_cond, dtype=float)).sum()),
      " score_cond:", score_cond)

oof_cond: (1661,)  NaN: 0  score_cond: 1.0706039569053587


In [6]:
# ===== チェックポイント 1: OOF 予測 =====
check("A-1 oof_cond の長さ", arr_stat(oof_cond, "len"), 1661.0,
      hint="len(train) と同じ長さの配列を np.full(len(train), np.nan) で作る。")

check("A-2 NaN が残っていないか", arr_stat(oof_cond, "nan"), 0.0,
      hint="NaN が残る = その行が一度も検証側に入っていない。kf.split(train) を全部回しているか確認。")

check("A-3 oof_cond の平均", arr_stat(oof_cond, "mean"), 9.02885435493378,
      hint="fit_predict_mean が返すのは (予測, 未知件数) のタプル。pred, _ = ... のように2つに受けているか確認。")

check("A-4 予測値の種類数", arr_stat(oof_cond, "nunique"), 20.0,
      hint="condition は4種類 × fold 5回 = 最大20通りの平均値ができる。1種類しかないなら key を間違えている。")

check("A-5 score_cond", score_cond, 1.0706039569053587,
      hint="rmse_log(oof_cond, y)。0.74 付近なら key が 'category' のまま。")

check("A-6 score_cond は Python の float か",
      1.0 if isinstance(score_cond, float) else 0.0, 1.0,
      hint="rmse_log は float を返すのでそのまま入れれば OK。np.float64 になっているなら float(...) で包む。")

print("\n(6つとも [OK] になったら次の概念へ)")
print("補足: condition だけのモデルは 1.07、category だけのモデルは 0.75。")
print("      『どの列で群を作るか』でここまで違う。だが今日の主題はモデルではなく検証だ。")

[OK] A-1 oof_cond の長さ: 正解!
[OK] A-2 NaN が残っていないか: 正解!
[OK] A-3 oof_cond の平均: 正解!
[OK] A-4 予測値の種類数: 正解!
[OK] A-5 score_cond: 正解!
[OK] A-6 score_cond は Python の float か: 正解!

(6つとも [OK] になったら次の概念へ)
補足: condition だけのモデルは 1.07、category だけのモデルは 0.75。
      『どの列で群を作るか』でここまで違う。だが今日の主題はモデルではなく検証だ。


---
# 概念2 — グループリーク(今日の山場)

## ① なぜ: 同じ商品が、複数のサイトから重複して集まってくる

君の会社のように Web からデータを集めていると、これは避けられない。
**同じ商品**が A社のサイトにも B社のサイトにも載っていて、両方をクロールすれば**ほぼ同一の行が2つ**できる。
価格もほぼ同じだ(同じ商品なのだから当たり前)。

ここでランダムに train / valid を分割すると何が起きるか。

> 同じ商品の1行目が **train** に、2行目が **valid** に入る。
> モデルは train 側でその商品の価格をすでに見ている。valid でそれを当てても、**何の実力も示していない**。

これは Web データに限らない。**同じ患者の複数の検査**、**同じユーザーの複数のセッション**、
**同じ工場ラインの連続したロット** — 「1つの実体から複数の行が生まれる」データは全部これになる。
そして厄介なことに、**CV スコアは上がるので気づけない**。気づくのは本番に出した後だ。

今日はこれを、**モデルもデータも1文字も変えず、分割方法だけを変えて**数値で見る。

## ② 解説: group を決めるのは「本番で何が未知か」

### まずデータの生成過程を確認する

| 列 | 何者か |
|---|---|
| `product_key` | **商品の識別子**。同じ商品なら同じ値(= これが「同じ実体」の正体) |
| `model_code` | 型番。`product_key` とほぼ1対1に対応する |
| `site` | 収集元のサイト(`site_A`〜`site_D`) |
| `record_id` | 行の識別子。全行ユニークなので**グループにはならない** |

現実のコンペでは `product_key` のような親切な列は無いことが多い。
「どれとどれが同じ商品か」を表記ゆれのある商品名から自分で作る作業が**名寄せ**で、それを unit06 でやる。
今日は「名寄せ済みのキーがある」状態からスタートする。

### `GroupKFold` — 同じグループを train と valid に跨らせない

```
product_key:  P1 P1 P2 P3 P3 P3 P4 ...

KFold(ランダム):        [P1][P3][P2]  |  [P1][P3][P3][P4]     ← P1 も P3 も両側にいる = リーク
GroupKFold(P で分ける):  [P1][P1][P3][P3][P3]  |  [P2][P4]     ← どのグループも片側だけ
```

```python
gkf = GroupKFold(n_splits=5)
for tr_idx, va_idx in gkf.split(train, y, groups=train["product_key"]):
    ...
```

`split()` の**第3引数 `groups`** に「各行がどのグループか」を並べた配列を渡す。
`KFold` と**まったく同じインターフェース**(C# で言えば同じインターフェースの別実装)なので、
呼び出し側のループは1文字も変えずに差し替えられる。これが sklearn の設計思想で、unit03 で本格的に扱う。

> 制約が1つ: **`n_splits` はグループの種類数以下**でなければならない。
> グループが4種類しかないのに `GroupKFold(5)` を作ると `ValueError` になる。

### group に何を選ぶか — 判断基準は1つだけ

group キーは「同じ実体」を表すものなら何でもいい、ではない。決め方はこうだ。

> **本番(test)では何が未知なのかを見て、それを group にする。**

| 本番の状況 | group にすべきもの | 検証で再現される状況 |
|---|---|---|
| 既知のサイトに**新商品**が並ぶ | `product_key` | 「見たことのない商品を当てる」 |
| **新しいサイト**を追加してクロールする | `site` | 「見たことのないサイトのデータを当てる」 |
| 両方 | `StratifiedGroupKFold` や複合キー | — |

#### 判断の順番: 本番の問い → group キー → 検証

今回の本番は「**既知のサイトに並ぶ新商品**」の価格を当てること。したがって、検証でも「その商品を学習時に一度も見ていない」状態を作りたい。
そこで同じ商品を表す `product_key` を group にし、同じ `product_key` の行を丸ごと学習側か検証側のどちらかへ置く。

`model_code` は今回の**予測に使う列**、`product_key` は同じ実体を隔離するための**分割に使う列**だ。ほぼ1対1対応しているため、商品を丸ごと検証へ回せば、型番もほぼ未知になる。

本番の問いが「新しいサイトでも当てられるか」なら `site` を group にする。つまり group は『同じものを見分けるため』だけでなく、**本番で再現したい未知の単位を丸ごと隠すため**に選ぶ。

### 診断のための1つの数字: 「未知型番率」

グループリークが起きているかどうかは、**検証時に『学習で一度も見ていないキー』が何%あるか**で分かる。今回の型番平均モデルでは、次の割合である。

```text
未知型番率 = 検証側で学習側に存在しない model_code の行数 ÷ 検証行数
```

これは `GroupKFold` に渡す引数でも、96%などの目標値を直接指定するものでもない。**分割の結果が本番をどれくらい再現できているかを測る診断値**である。

目標にすべき未知率は、本番で何を予測するかによって変わる。既存商品の再販を当てる本番なら未知率は低くてよく、新商品を当てる本番なら高くなる。分類でも回帰でも、この考え方は同じ。

- 本番で 96% が未知なのに、検証では 20% しか未知でない → **検証が本番より簡単すぎる**
- `product_key` の GroupKFold で 100% 未知 → 本番の96%に近く、少し厳しめだが本番をかなり再現できている
- 本番と検証で未知率が近い → **その検証は本番を再現できている可能性が高い**

今日作る `fit_predict_mean` は「対応表に無かった行数」を返すようにしてある。この数字を毎回見る。

In [7]:
# GOAL: 「同じ型番なら同じ価格だろう」という妥当に見えるベースラインを、素朴な KFold で評価する

# STEP 1: データの生成過程を数字で見る — 1商品が何行あるか
vc = train["product_key"].value_counts()
print("--- 重複構造 ---")
print("  train の行数        :", len(train))
print("  ユニークな商品数    :", train["product_key"].nunique())
print("  1商品あたり行数の分布:", vc.value_counts().sort_index().to_dict(), " (行数 -> 商品数)")
print("  2行以上ある商品の割合:", round(float((vc >= 2).mean()), 3))

# STEP 2: 同じ商品の price はどれくらい揃っているか(揃っているほどリークが効く)
g = train.groupby("product_key")["price"]
cv_within = (g.std() / g.mean()).dropna()      # 変動係数 = 標準偏差 / 平均。0に近いほど「ほぼ同じ値」
print("\n  同一商品内の price 変動係数の中央値:", round(float(cv_within.median()), 4))
print("  → 同じ商品はサイトが違っても価格がほぼ同じ。片方を見れば、もう片方はほぼ当たる。")

# STEP 3: model_code(型番)は商品をほぼ一意に指している
print("\n  product_key の種類数:", train["product_key"].nunique())
print("  model_code  の種類数:", train["model_code"].nunique(), " ← ほぼ1対1")

# STEP 4: 素朴で妥当に見えるベースライン — 「型番ごとの平均価格を貼る」
#         未知の型番は全体平均で埋める。ここまでは何もおかしなことをしていない。
print("\n--- ベースライン: model_code ごとの平均 log price ---")
score_kf, unknown_kf, oof_kf = run_cv("model_code", KFold(n_splits=5, shuffle=True, random_state=0))
print(f"  KFold(5, shuffle=True) の CV スコア : {score_kf:.4f}")
print(f"  検証時の未知型番率                 : {unknown_kf:.1%}")
print(f"\n  参考: category 平均モデルの CV     : {rmse_log(oof, y):.4f}")
print("  → 型番を使うと 0.75 から 0.61 へ大きく改善した。素晴らしい特徴量に見える。")
print("    本当だろうか?")

--- 重複構造 ---
  train の行数        : 1661
  ユニークな商品数    : 756
  1商品あたり行数の分布: {1: 204, 2: 281, 3: 189, 4: 82}  (行数 -> 商品数)
  2行以上ある商品の割合: 0.73

  同一商品内の price 変動係数の中央値: 0.2348
  → 同じ商品はサイトが違っても価格がほぼ同じ。片方を見れば、もう片方はほぼ当たる。

  product_key の種類数: 756
  model_code  の種類数: 756  ← ほぼ1対1

--- ベースライン: model_code ごとの平均 log price ---
  KFold(5, shuffle=True) の CV スコア : 0.6078
  検証時の未知型番率                 : 20.4%

  参考: category 平均モデルの CV     : 0.7458
  → 型番を使うと 0.75 から 0.61 へ大きく改善した。素晴らしい特徴量に見える。
    本当だろうか?


## ④ 予測: モデルもデータも変えず、分割方法だけを変えたら?

③ で出た CV は **0.6078**、検証時の未知型番率は **20.4%**。
「型番ごとの平均を貼る」という、誰がやってもそう書きそうなベースラインだ。

次のセルでは、**モデルも特徴量もデータも1文字も変えない**。変えるのは分割方法だけ:

1. `KFold(5, shuffle=True)` → `GroupKFold(5)` に `groups=train["product_key"]` を渡す

実行する前に予測しよう。

1. **GroupKFold にしたときの CV スコアは、いくつくらいになる?** 0.61 から何割くらい悪化する?
   (0.62? 0.65? 0.8? 1.1?)
2. **GroupKFold のときの「未知型番率」は何%になる?** ヒント: `model_code` と `product_key` はほぼ1対1だ。
3. **本番LB(未来の test)での未知型番率**は何%だと思う?
   ヒント: `test` は train の後の月に収集されたデータで、共通する `product_key` はごくわずかしかない。
4. 3つのうち、**本番LB に最も近い値を出すのはどの検証方法**?

> 一度、紙かコメントに数字を書いてから実行してほしい。この差の大きさが今日の山場だ。

In [8]:
# GOAL: 「未知型番率」という1つの数字で、CVと本番のズレが全部説明できることを見る

# STEP 1: 分割方法だけを GroupKFold に差し替える。ループの中身は run_cv のまま = 1文字も変えていない。
score_gkf, unknown_gkf, oof_gkf = run_cv("model_code", GroupKFold(n_splits=5), groups=train["product_key"])

# STEP 2: 本番LB — train 全部で学習して、未来の test を予測する
table_full = pd.Series(y).groupby(train["model_code"]).mean()   # train 全体で作った「型番 -> 平均」
pred_test = test["model_code"].map(table_full)
unknown_lb = float(pred_test.isna().mean())                     # test の型番のうち、train に無かった割合
score_lb = lb_score(pred_test.fillna(float(y.mean())).to_numpy(dtype=float))

# STEP 3: 3つを並べる
print(f"{'検証方法':<34}{'RMSLE':>10}{'未知型番率':>14}")
print("-" * 58)
print(f"{'KFold(5, shuffle=True)':<32}{score_kf:>10.4f}{unknown_kf:>14.1%}")
print(f"{'GroupKFold(5, product_key)':<32}{score_gkf:>10.4f}{unknown_gkf:>14.1%}")
print(f"{'本番LB(未来の test)':<28}{score_lb:>10.4f}{unknown_lb:>14.1%}")

print("\n--- 読み方 ---")
print(f"  KFold は本番の {score_kf / score_lb:.0%} のスコアを報告していた。つまり実力を2倍近く良く見せていた。")
print(f"  GroupKFold は本番 {score_lb:.4f} に対して {score_gkf:.4f}。ほぼ的中している。")
print("\n  原因は右の列だけで説明できる:")
print("    本番では 96% の型番が『初めて見るもの』なのに、")
print("    KFold の検証では 80% が『train 側で答えを見た型番』になっていた。")
print("    同じ商品が複数サイトから重複収集されているので、片方が train、片方が valid に割れて入るためだ。")

# STEP 4: 本当にそうなっているか、1商品を実際に追いかけて確かめる
dup = vc[vc >= 2].index[0]
rows = train.loc[train["product_key"] == dup, ["record_id", "product_key", "model_code", "site", "collected_at", "price"]]
print(f"\n--- 重複の実物: 商品 {dup} ---")
print(rows.to_string(index=False))
pos = np.where(train["product_key"].to_numpy() == dup)[0]
for fold, (tr_idx, va_idx) in enumerate(KFold(n_splits=5, shuffle=True, random_state=0).split(train)):
    in_tr = sorted(set(pos) & set(tr_idx.tolist()))
    in_va = sorted(set(pos) & set(va_idx.tolist()))
    if in_tr and in_va:
        print(f"  KFold fold {fold}: この商品の行 {in_tr} が train、{in_va} が valid に割れている ← リーク発生")
        break
print("  → GroupKFold ならこの割れ方は原理的に起きない。")

検証方法                                   RMSLE         未知型番率
----------------------------------------------------------
KFold(5, shuffle=True)              0.6078         20.4%
GroupKFold(5, product_key)          1.1119        100.0%
本番LB(未来の test)                  1.1670         96.4%

--- 読み方 ---
  KFold は本番の 52% のスコアを報告していた。つまり実力を2倍近く良く見せていた。
  GroupKFold は本番 1.1670 に対して 1.1119。ほぼ的中している。

  原因は右の列だけで説明できる:
    本番では 96% の型番が『初めて見るもの』なのに、
    KFold の検証では 80% が『train 側で答えを見た型番』になっていた。
    同じ商品が複数サイトから重複収集されているので、片方が train、片方が valid に割れて入るためだ。

--- 重複の実物: 商品 P00845 ---
record_id product_key   model_code   site collected_at  price
   R00002      P00845 MC-3486-0845 site_C   2025-12-09   3993
   R00202      P00845 MC-3486-0845 site_D   2025-12-09   4542
   R01752      P00845 MC-3486-0845 site_B   2025-12-07   3264
   R01888      P00845 MC-3486-0845 site_A   2025-12-11   3447
  KFold fold 0: この商品の行 [np.int64(0), np.int64(174), np.int64(1465)] が train、[np.int64(1573)] が valid に割れている ← リーク発生
 

## ⑥ 書いてみる: group キーを間違えるとどうなるか

②で「group は**本番で何が未知か**から決める」と書いた。これを実際に外してみる。

今回の本番は「**既知のサイトに、見たことのない新商品が並ぶ**」状況だ。だから正解は `product_key`。
では、うっかり **`site`(収集元サイト)を group にしてしまった**らどうなるだろうか。
`site` を group にするというのは「**新しいサイトを追加でクロールする**」状況を検証していることになる。

次のセルで、`run_cv` を使って以下の2つを求めよう。

- `score_site_group` — `model_code` 平均モデルを **`GroupKFold(n_splits=4)`** で、
  **`groups=train["site"]`** として交差検証したときの CV スコア(float)
- `unknown_site_group` — そのときの**未知型番率**(float、0〜1の割合)

`run_cv` は **`(スコア, 未知キー率, OOF配列)` の3つ組**を返す。3つに受けて先頭2つを使えばいい。2〜3行で書ける。

> **なぜ `n_splits=4` なのか**: `site` は `site_A`〜`site_D` の**4種類しかない**。
> `GroupKFold(5)` を作ると「グループ数より分割数が多い」で `ValueError` になる。
> グループの種類数が分割数の上限になる、というのは GroupKFold を使うときに毎回引っかかる制約だ。

書けたら、③⑤ で出た3つの数字(0.6078 / 1.1119 / 1.1670)と並べて眺めてみよう。

In [9]:
score_site_group = None
unknown_site_group = None
# ここに書く(ヒント: run_cv("model_code", GroupKFold(n_splits=4), groups=train["site"]) の戻り値は3つ組)
score_site_group,unknown_site_group,_ = run_cv("model_code", GroupKFold(n_splits=4), groups=train["site"])

print("score_site_group:", score_site_group, " unknown_site_group:", unknown_site_group)

score_site_group: 0.5168911224381422  unknown_site_group: 0.12281757977122215


In [10]:
# ===== チェックポイント 2: group キーの選択 =====
check("B-1 score_site_group", score_site_group, 0.5168911224381422,
      hint="run_cv('model_code', GroupKFold(n_splits=4), groups=train['site']) の1つめの戻り値。"
           "1.11 になったなら groups に product_key を渡している。")

check("B-2 unknown_site_group(未知型番率)", unknown_site_group, 0.12281757977122215,
      hint="run_cv の2つめの戻り値。パーセントではなく 0〜1 の割合。")

check("B-3 site を group にしたスコアは KFold より良い(= さらに甘い)か",
      1.0 if (isinstance(score_site_group, float) and score_site_group < 0.6078060801986365) else 0.0, 1.0,
      hint="score_site_group が正しく入っていれば自動で [OK] になる。")

print("\n--- この結果の読み方 ---")
print("  site を group にすると、未知型番率は 20.4% -> 12.3% と『下がって』しまった。")
print("  各サイトの商品は他サイトとかなり重なっているので、1サイトを valid に回しても")
print("  その商品の答えは残り3サイト(= train 側)に載っている。リークは全く潰れていない。")
print("  スコアも 0.61 -> 0.52 とさらに良くなり、本番 1.17 からますます遠ざかった。")
print("\n  教訓: GroupKFold を使えば安全、ではない。**何を group にするか**が全て。")
print("        判断基準は1つ — 本番(test)で未知になるものを group にする。")

[OK] B-1 score_site_group: 正解!
[OK] B-2 unknown_site_group(未知型番率): 正解!
[OK] B-3 site を group にしたスコアは KFold より良い(= さらに甘い)か: 正解!

--- この結果の読み方 ---
  site を group にすると、未知型番率は 20.4% -> 12.3% と『下がって』しまった。
  各サイトの商品は他サイトとかなり重なっているので、1サイトを valid に回しても
  その商品の答えは残り3サイト(= train 側)に載っている。リークは全く潰れていない。
  スコアも 0.61 -> 0.52 とさらに良くなり、本番 1.17 からますます遠ざかった。

  教訓: GroupKFold を使えば安全、ではない。**何を group にするか**が全て。
        判断基準は1つ — 本番(test)で未知になるものを group にする。


---
# 概念3 — 時間リーク と TimeSeriesSplit

## ① なぜ: 本番はいつも「過去で学習して、未来を予測する」

このコンペの `train` は 2025-10-01〜2026-02-28、`test` は 2026-03-01 以降だ。
本番でやらされるのは「**過去5か月のデータから、来月を当てる**」こと。

ところがランダムな KFold は「**2月のデータで学習して、10月を予測する**」形も平気で作る。
未来を見てから過去を当てているわけで、本番では絶対に起こらない状況だ。しかも**簡単な**状況だ。

実務ではこちらの方が主戦場になる。需要予測、解約予測、価格予測 — ほぼ全部が「未来を当てる」形をしている。
そして相場・トレンド・季節性がある限り、**未来を見られるかどうか**は本当に効く。
このデータの価格は**月あたり約6%下落する**ように作ってある(教材なので生成過程を明かす)。

## ② 解説: TimeSeriesSplit と、その特有の落とし穴

### 図で見る TimeSeriesSplit

```
時間順に並べたデータ →  [ブロック0][ブロック1][ブロック2][ブロック3][ブロック4][ブロック5]

fold 0:                 [  学習  ][  検証  ]
fold 1:                 [  学習  ][  学習  ][  検証  ]
fold 2:                 [  学習  ][  学習  ][  学習  ][  検証  ]
fold 3:                 [  学習  ][  学習  ][  学習  ][  学習  ][  検証  ]
fold 4:                 [  学習  ][  学習  ][  学習  ][  学習  ][  学習  ][  検証  ]
                          ↑
                    ブロック0 は一度も「検証」にならない
```

学習期間がだんだん伸びていくので **expanding window(拡張窓)** と呼ばれる。
「常に過去だけで学習し、必ずその先の期間で検証する」— 本番と同じ形になる。

### 落とし穴1: **データが時間順に並んでいないと意味がない**

`TimeSeriesSplit` は日付列を見ない。**行の並び順を時間順だと信じて**先頭から順に切るだけだ。
`train.csv` はシャッフル済みなので、そのまま渡すと**ただのランダム分割**になってしまう。
使う前に必ず `sort_values("collected_at")` する。

### 落とし穴2: **全行は検証されない** ← ここを実際に踏む

上の図の通り、**ブロック0 は一度も検証側にならない**。
`TimeSeriesSplit(5)` は 1661行を6ブロックに割るので、先頭の 281 行は検証されず、
**検証されるのは 1380 行だけ**になる。

ここで OOF 配列を `np.zeros(n)` で初期化していると、**検証されなかった行に 0 が残ったまま**になる。
そのまま全行で採点すると、「予測値 0 = log1p(価格) が 0 = 価格が0円」という
**あり得ない予測を281件出した**ことになって、スコアが出鱈目な値になる。

```python
oof = np.zeros(n)                 # ← 危険。未検証の行が「0 という予測」に化ける
oof = np.full(n, np.nan)          # ← 正しい。「予測なし」を NaN で表す
score = rmse_log(oof[~np.isnan(oof)], y[~np.isnan(oof)])   # 埋まった行だけで採点
```

`~np.isnan(oof)` は既習の numpy-masking そのもの。③ で「間違ったやり方」と「正しいやり方」を並べて出す。

### 今日ここで使う LightGBM

概念3・4 では、群平均モデルより少し強いモデルが要る。**LightGBM** を使うが、
**今日は「検証の道具」としてだけ**使い、パラメータの意味は unit03 でやる。ひとまずこれで固定する:

```python
LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)
```

`verbose=-1` は「学習中の大量のログを黙らせる」指定。fit / predict は sklearn の他のモデルと共通の作法だ。

### 特徴量を作るときの pandas 3.0 の罠

「カテゴリ列(文字列の列)を探す」というのは前処理で必ず書くコードだが、
ネット記事の定番の書き方が **pandas 3.0 では空振りする**。

| 判定方法 | pandas 1.x / 2.x | pandas 3.0(この環境) |
|---|---|---|
| `df[c].dtype == object` | 文字列列が引っかかる | **何も引っかからない**(文字列の dtype が `str` になったため) |
| `not pd.api.types.is_numeric_dtype(df[c])` | 引っかかる | **引っかかる**(こちらが正解) |

③ の STEP 1 で実物を見る。unit01 で見た `select_dtypes(include="object")` の空振りと同じ根っこの問題だ。

In [ ]:
# GOAL: ランダム分割と時系列分割で、同じモデルのスコアがどう変わるか。そして TimeSeriesSplit の罠を踏む

# STEP 1: カテゴリ列を探す — pandas 3.0 の罠を実物で確認する
print("--- カテゴリ列の判定(pandas", pd.__version__, ")---")
print("  dtype == object で判定      :", [c for c in train.columns if train[c].dtype == object], " ← 空振り!")
print("  is_numeric_dtype の否定で判定:", [c for c in train.columns if not pd.api.types.is_numeric_dtype(train[c])])
print("  → 後者が正解。ただし日時列(collected_at)も『数値でない』ので混ざる点に注意。使う列は明示的に選ぶ。")

# STEP 2: 特徴量を作る。model_code / product_key は使わない(概念2 で『使うと本番で効かない』と分かった)
TRAIN_START = train["collected_at"].min()
train_f = train.copy()
test_f = test.copy()
for d in (train_f, test_f):
    d["days"] = (d["collected_at"] - TRAIN_START).dt.days     # 起点からの経過日数(unit04 で本格的にやる日付特徴)

NUM_COLS = ["brand_tier", "views", "title_len", "days"]
CAT_COLS = ["category", "site", "condition"]
CAT_LEVELS = {c: sorted(train[c].unique()) for c in CAT_COLS}   # 水準は train で固定する(test だけの値に振り回されない)


def make_features(df, extra=()):
    """特徴量の DataFrame を作る。extra には追加で入れたい列名を渡す(概念4で使う)。"""
    X = df[NUM_COLS].copy()
    for c in CAT_COLS:
        # pandas の category dtype は C# の enum に近い「取りうる値が決まっている型」。
        # LightGBM は category dtype の列を自動でカテゴリとして扱ってくれる(数値の大小として誤解しない)。
        X[c] = pd.Categorical(df[c].to_numpy(), categories=CAT_LEVELS[c])
    for c in extra:
        X[c] = df[c].to_numpy()
    return X


X_train = make_features(train_f)
print("\n--- 特徴量 ---")
print("  X_train.shape:", X_train.shape, " (train と同じ行数)")
print(X_train.dtypes.to_string())


# STEP 3: LightGBM で交差検証する関数。中身の構造は run_cv とまったく同じ。
def run_lgb_cv(X, target, splitter, groups=None):
    """LightGBM を splitter で交差検証し、OOF 配列を返す(未検証の行は NaN のまま)。"""
    oof = np.full(len(X), np.nan)
    for tr_idx, va_idx in splitter.split(X, target, groups):
        model = LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)
        model.fit(X.iloc[tr_idx], target[tr_idx])
        oof[va_idx] = model.predict(X.iloc[va_idx])
    return oof


# STEP 4: まずランダムな KFold
oof_random = run_lgb_cv(X_train, y, KFold(n_splits=5, shuffle=True, random_state=0))
score_random = rmse_log(oof_random, y)
print(f"\nKFold(5, shuffle=True) の CV : {score_random:.4f}   (検証された行数 {int((~np.isnan(oof_random)).sum())})")

# STEP 5: TimeSeriesSplit は「行の並び順」を時間順だと信じる。必ずソートしてから渡す。
train_sorted = train_f.sort_values("collected_at").reset_index(drop=True)
y_sorted = np.log1p(train_sorted["price"].to_numpy(dtype=float))
X_sorted = make_features(train_sorted)

print("\n--- TimeSeriesSplit(5) の分割構造 ---")
tss = TimeSeriesSplit(n_splits=5)
for fold, (tr_idx, va_idx) in enumerate(tss.split(X_sorted)):
    tr_dates = train_sorted["collected_at"].iloc[tr_idx]
    va_dates = train_sorted["collected_at"].iloc[va_idx]
    print(f"  fold {fold}: train {len(tr_idx):>4}行 [{tr_dates.min().date()} .. {tr_dates.max().date()}]"
          f"  valid {len(va_idx):>4}行 [{va_dates.min().date()} .. {va_dates.max().date()}]")

oof_time = run_lgb_cv(X_sorted, y_sorted, tss)
validated = ~np.isnan(oof_time)
print(f"\n  検証された行数: {int(validated.sum())} / {len(train_sorted)}"
      f"  ← 先頭 {len(train_sorted) - int(validated.sum())} 行はどの fold でも検証されていない")

# STEP 6: 【罠】未検証の行を 0 のまま採点するとどうなるか
oof_zeros = np.nan_to_num(oof_time, nan=0.0)     # np.zeros で初期化したまま回したのと同じ状態
score_wrong = rmse_log(oof_zeros, y_sorted)
score_right = rmse_log(oof_time[validated], y_sorted[validated])
print("\n--- OOF 配列の初期化で結果がここまで変わる ---")
print(f"  np.zeros で初期化 → 全 {len(train_sorted)} 行で採点 : {score_wrong:.4f}   ← 出鱈目")
print(f"  np.full(nan)で初期化 → 埋まった {int(validated.sum())} 行だけで採点: {score_right:.4f}   ← 正しい")
print("  0 という予測は log1p(価格)=0、つまり『価格0円』を281件出したのと同じ扱いになる。")
print("  スコアが桁違いにおかしいときは、まず OOF に未記入の行が無いかを疑う。")

## ④ 予測: どちらの検証が本番を言い当てる?

③ で2つの数字が出た。

| 検証方法 | CV |
|---|---|
| `KFold(5, shuffle=True)` | **0.6859** |
| `TimeSeriesSplit(5)`(日付順にソート済み) | **0.7145** |

次のセルでは、この特徴量・このモデルのまま **本番LB(未来の test を当てる)** を実測する。実行前に予測しよう。

1. **本番LB はいくつくらいになる?** 0.69 に近い? 0.71 に近い? それとももっと悪い?
2. 概念2 では KFold が本番より**大幅に甘かった**。今回の差(0.686 vs 0.714)は概念2 の差(0.61 vs 1.17)より
   ずっと小さい。**なぜ小さいのだろう?**(ヒント: 今回の特徴量には `model_code` が入っていない)
3. **`sort_values("collected_at")` を忘れて、シャッフルされたままの `X_train` に `TimeSeriesSplit` を掛けたら**
   スコアはどうなる? 時系列分割なのだから、ソートしなくても時系列っぽい結果になる?

> 3 は実際によくやるミスだ。`TimeSeriesSplit` を書いた時点で安心してしまって、
> 並び順を確認し忘れる。しかもエラーにはならない — 静かに間違った数字が出る。

In [ ]:
# GOAL: 3つの数字を並べて「どの検証が本番を言い当てたか」を確認し、ソート忘れの害を見る

# STEP 1: 本番LB — train 全部で学習して未来の test を予測する
model_full = LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)
model_full.fit(X_train, y)
pred_test_lgb = model_full.predict(make_features(test_f))
score_lb_lgb = lb_score(pred_test_lgb)

print(f"{'検証方法':<38}{'RMSLE':>10}{'本番との差':>12}")
print("-" * 60)
print(f"{'KFold(5, shuffle=True)':<36}{score_random:>10.4f}{score_random - score_lb_lgb:>12.4f}")
print(f"{'TimeSeriesSplit(5)':<36}{score_right:>10.4f}{score_right - score_lb_lgb:>12.4f}")
print(f"{'本番LB(未来の test)':<32}{score_lb_lgb:>10.4f}{0.0:>12.4f}")

print("\n--- 読み方 ---")
print("  KFold は本番より 0.02 甘い(未来を見てから過去を当てている分だけ楽をしている)。")
print("  TimeSeriesSplit は本番より 0.008 辛い。実務ではこの向きのズレの方がずっと安全 --")
print("  『手元で見た数字より本番が良かった』は誰も困らないが、逆は事故になる。")
print("\n  概念2 ほど差が開かなかったのは、今回の特徴量に model_code を入れていないから。")
print("  リークの大きさは『どんな検証をしたか』だけでなく『どんな特徴量を使ったか』でも決まる。")

# STEP 2: 【罠】ソートを忘れて TimeSeriesSplit を掛けるとどうなるか
oof_unsorted = run_lgb_cv(X_train, y, TimeSeriesSplit(n_splits=5))   # X_train はシャッフルされたまま
m_uns = ~np.isnan(oof_unsorted)
score_unsorted = rmse_log(oof_unsorted[m_uns], y[m_uns])
print("\n--- ソート忘れ ---")
print(f"  ソートしてから TimeSeriesSplit : {score_right:.4f}")
print(f"  ソートせずに TimeSeriesSplit   : {score_unsorted:.4f}   ← KFold(0.6859)寄りの甘い値に戻ってしまった")
print("  エラーは1つも出ない。TimeSeriesSplit は日付列を見ず、行の並び順だけを信じるため。")

# STEP 3: 時間による下落は生のデータでは見えにくい、という現実も確認しておく
train_f["log_price"] = y
month = train_f["collected_at"].dt.to_period("M")
print("\n--- 月別の log price ---")
raw = train_f.groupby(month)["log_price"].mean()
# 群平均を「元の行数のまま」貼り戻す transform('mean')(unit04 で本格的に扱う)。
# カテゴリ・状態・ブランド帯の違いを差し引いた残りだけを見る = 交絡を取り除く。
base = train_f.groupby(["category", "condition", "brand_tier"])["log_price"].transform("mean")
adj = (train_f["log_price"] - base).groupby(month).mean()
print(pd.DataFrame({"生の平均": raw.round(4), "商品構成を揃えた後": adj.round(4)}).to_string())
print("\n  生の平均は上下していて、下落トレンドが見えない(月ごとに売れ筋カテゴリの構成が違うため)。")
print("  構成を揃えると +0.17 → -0.06 と単調に下がっている。これが月6%の相場下落の正体。")
print("  → 『目で見て分からないリーク』はいくらでもある。だから分割方法で守る。")

## ⑥ 書いてみる: 日付で切るホールドアウトを自分で作る

`TimeSeriesSplit` は便利だが、実務では「**最後の1か月を検証に使う**」のような
**日付を指定した手動の分割**の方がよく使われる。理由は明快で、本番の予測期間の長さに合わせられるからだ
(来月を予測するなら、検証も1か月ぶんで測りたい)。

次のセルで、`2026-02-01` を境にした分割を作ろう。

- `tr_idx_time` — `collected_at` が **`2026-02-01` より前**の行の**位置インデックス**(int の配列)
- `va_idx_time` — `collected_at` が **`2026-02-01` 以降**の行の位置インデックス
- `score_time_holdout` — `X_train` と `y` を使い、
  `LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)` を `tr_idx_time` で学習して
  `va_idx_time` を予測したときの `rmse_log`(float)

使う道具:

| やること | 書き方 |
|---|---|
| 日付の比較用の値 | `cut = pd.Timestamp("2026-02-01")` |
| 条件を満たす行の**位置** | `np.where(train["collected_at"] < cut)[0]` |
| 位置で行を取る | `X_train.iloc[tr_idx_time]`、`y[tr_idx_time]` |

> `np.where(条件)` は「True の位置の配列」をタプルで返すので、`[0]` で中身を取り出す。
> 既習のブールマスクから位置インデックスへの変換だ(`train[mask]` ではなく**位置**が欲しいのは、
> `.iloc` と `y[...]` の両方に同じものを使いたいから)。

5〜7行で書ける。出た数字を、⑤ の3つ(0.6859 / 0.7145 / 0.7060)と比べてみよう。**なぜズレるのか**も考えてほしい。

In [ ]:
cut = pd.Timestamp("2026-02-01")
tr_idx_time = None
va_idx_time = None
score_time_holdout = None
# ここに書く(ヒント: np.where(train["collected_at"] < cut)[0] で位置を取り、
#           LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1).fit(X_train.iloc[tr_idx_time], y[tr_idx_time]))

print("train:", None if tr_idx_time is None else len(tr_idx_time),
      " valid:", None if va_idx_time is None else len(va_idx_time),
      " score:", score_time_holdout)

In [ ]:
# ===== チェックポイント 3: 日付ホールドアウト =====
check("C-1 学習に使う行数(2026-02-01 より前)", arr_stat(tr_idx_time, "len"), 1411.0,
      hint="np.where(train['collected_at'] < cut)[0] の長さ。1661 のままなら条件が効いていない。")

check("C-2 検証に使う行数(2026-02-01 以降)", arr_stat(va_idx_time, "len"), 250.0,
      hint="境界は『以降』なので >= を使う。< と <= を取り違えると1〜数行ずれる。")

check("C-3 2つを足すと train 全行になるか",
      (arr_stat(tr_idx_time, "len") or 0) + (arr_stat(va_idx_time, "len") or 0), 1661.0,
      hint="重なりも抜けも無いように、同じ cut で < と >= に分ける。")

check("C-4 score_time_holdout", score_time_holdout, 0.6620258941509336,
      hint="X_train(ソートしていない方)と y をそのまま位置インデックスで切る。"
           "X_sorted を使うと行の対応がずれるので注意。")

print("\n--- 出た数字の読み方 ---")
print("  日付ホールドアウト 0.6620 は、TimeSeriesSplit の 0.7145 とも本番 0.7060 とも違う。")
print("  『未来を検証している』のは同じなのに、なぜズレるのか?")
print("  → 検証が 250 行の1回きりだから。概念1 で見たホールドアウトのばらつきがそのまま出ている。")
print("    だから TimeSeriesSplit は『未来検証を複数回やって均す』形になっている。")
print("    時間分割にしただけでは足りず、回数も要る — 概念1 と概念3 は同じことを言っている。")

---
# 概念4 — ターゲットリーク(良すぎるスコアを疑う)

## ① なぜ: 「精度99%出ました」の9割はリーク

実務でこの報告が上がってきたとき、ベテランがまずやるのは喜ぶことではなく**特徴量の一覧を見ること**だ。
古典的な事故はどれも同じ形をしている。

- 解約予測モデルに **`解約日`** が入っていた(解約しない人は欠損なので、欠損かどうかで100%当たる)
- 受注予測に **`受注金額`** が入っていた
- 医療診断に **`処方された薬の名前`** が入っていた(その病気と診断されたから処方されている)
- 価格予測に **`割引率`** が入っていた ← **今日のデータがこれ**

共通するのは「**予測したい時点では、まだ手に入らないはずの情報**」が特徴量に混ざっていること。
これが**ターゲットリーク**だ。学習も検証も何の問題も起こさず、スコアだけが不自然に良くなる。
そして本番で列が手に入らない、あるいは値が入っていなくて、**モデルは何も予測できない**。

## ② 解説: 3つの検出方法

### このデータに仕込まれているもの

`train.csv` には `list_price`(定価)と `discount_rate`(割引率)がある。そして `discount_rate` は、

$$\text{discount\_rate} = \frac{\text{list\_price} - \text{price}}{\text{list\_price}}$$

として作られた列だ。式を変形すると、

$$\text{price} = \text{list\_price} \times (1 - \text{discount\_rate})$$

つまり **2列そろえば `price` が完全に復元できる**。これを特徴量に入れたモデルは、
予測しているのではなく**答えを計算している**。

### 検出方法1: 目的変数との相関が異常に高い列を探す

`np.corrcoef(x, y)` で相関係数を出す。目的変数と 0.9 以上の相関がある列は、まず疑う。
ただし**この方法だけでは `discount_rate` は見つからない**(単独では相関がほぼ0)。
リークは**組み合わせ**で起きることがある、というのが今日の学びの一つ。

### 検出方法2: 特徴量重要度が1〜2列に集中していないか

`model.feature_importances_` は「どの列がどれだけ分岐に使われたか」。
1列で重要度の大半を持っていったら、その列を見る。unit03 で詳しくやる。

### 検出方法3: **その列は、予測する時点で本当に手に入るのか** ← これが本命

技術的な検出の前に、まずこれを問う。そして今日のデータには**最大のヒント**がある。

> **`test.csv` に `discount_rate` 列が無い。**

コンペの作者が「この列は予測時点では手に入らない」と言ってくれているのと同じだ。
**train にあって test に無い列**は、目的変数か、リーク列か、そのどちらかしかない。

| 検出方法 | 見つかるもの | 限界 |
|---|---|---|
| 1. 目的変数との相関 | 単独で目的変数を説明する列 | 組み合わせのリークは見つからない |
| 2. 特徴量重要度の集中 | 実際に効きすぎている列 | 「本当に良い特徴量」と区別がつかない |
| 3. **train/test の列の差分・列の意味** | **予測時点で存在しない情報** | 自動化しづらい(が最も確実) |

### 今回の比較は GroupKFold で固定する

概念2 で「この題材の正しい検証は `GroupKFold(groups=product_key)`」と分かった。
ここからは**検証方法を GroupKFold に固定**して、**特徴量だけ**を動かす。
これが正しい実験の作法だ — **一度に1つだけ変える**。

In [ ]:
# GOAL: リーク列を1つ足すだけで CV が「良すぎる値」になることを、正しい検証の上で見る

# STEP 1: まず式を確かめる。price は本当に復元できるのか?
restored = train["list_price"] * (1 - train["discount_rate"])
print("--- price = list_price * (1 - discount_rate) ---")
print(pd.DataFrame({
    "price": train["price"].head(4),
    "list_price": train["list_price"].head(4),
    "discount_rate": train["discount_rate"].head(4).round(4),
    "復元値": restored.head(4).round(2),
}).to_string(index=False))
print("  最大誤差:", round(float((restored - train["price"]).abs().max()), 3), "円")
print("  → 丸め誤差の範囲で完全に一致する。この2列は price そのもの。")

# STEP 2: test に何が無いか(最大のヒント)
print("\n--- train にあって test に無い列 ---")
print(" ", sorted(set(train.columns) - set(test.columns)))
print("  price は目的変数なので当然。discount_rate が無いのは『予測時点では手に入らない』という宣言。")

# STEP 3: 検証方法を GroupKFold に固定して、特徴量だけを変える
gkf = GroupKFold(n_splits=5)
groups = train["product_key"]

oof_clean = run_lgb_cv(X_train, y, gkf, groups=groups)
score_clean = rmse_log(oof_clean, y)

X_leak = make_features(train_f, extra=["list_price", "discount_rate"])
oof_leak = run_lgb_cv(X_leak, y, gkf, groups=groups)
score_leak = rmse_log(oof_leak, y)

print("\n--- GroupKFold(5) 固定、特徴量だけを変える ---")
print(f"  リーク列なし                        : {score_clean:.4f}")
print(f"  リーク列あり(list_price, discount_rate): {score_leak:.4f}   ← 良すぎる")
print(f"  改善率: {(1 - score_leak / score_clean):.0%}")
print("\n  RMSLE 0.068 は『価格を平均7%の誤差で当てている』という意味。")
print("  中古品の実売価格をそんな精度で当てられるモデルは存在しない。喜ぶ前に疑う。")

# STEP 4: 検出方法2 — 重要度が集中していないか
model_leak = LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)
model_leak.fit(X_leak, y)
imp = pd.Series(model_leak.feature_importances_, index=X_leak.columns).sort_values(ascending=False)
print("\n--- 特徴量重要度(割合)---")
print((imp / imp.sum()).round(4).to_string())
print(f"  上位2列だけで {float((imp / imp.sum()).head(2).sum()):.0%} を占めている。")

## ④ 予測: リークは「2列そろって」初めて起きる?

③ では `list_price` と `discount_rate` を**両方**入れて 0.0681 になった。
`price = list_price × (1 - discount_rate)` なので、**2列そろわないと復元できない**はずだ。

次のセルでは、片方ずつ入れて試す。実行前に予測しよう。

1. **`list_price` だけ**を足したら CV はどうなる? リークなしの 0.7138 とほぼ同じ? それとも改善する?
2. **`discount_rate` だけ**を足したら? こちらはどうか?
3. **目的変数との相関**を計算したとき、`list_price` と `discount_rate` のどちらが高い?
   その相関だけを見てリーク列を探すと、**見逃すのはどちらの列**?
4. `list_price`(定価)は本当に「予測時点で手に入らない」列だろうか?
   test にはこの列が**ある**。これはリーク列と呼ぶべきか、それとも正当な特徴量か?

> 4 は答えが1つに決まらない問いだ。判断基準は「**本番の運用でこの値がいつ手に入るか**」。
> 定価は商品が出品された時点で分かるので、**実売価格を予測するときには使える**。
> ただし「定価が分かるほど強い情報なら、そもそも予測タスクとして成立しているのか」は別途考える必要がある。

In [ ]:
# GOAL: リークが「単独の列」ではなく「列の組み合わせ」で起きること、相関だけでは見つからないことを確認する

# STEP 1: 片方ずつ足して比べる(検証は GroupKFold のまま固定)
rows = []
for label, extra in [("リーク列なし", ()),
                     ("+ list_price のみ", ("list_price",)),
                     ("+ discount_rate のみ", ("discount_rate",)),
                     ("+ 両方", ("list_price", "discount_rate"))]:
    Xi = make_features(train_f, extra=extra)
    oof_i = run_lgb_cv(Xi, y, GroupKFold(n_splits=5), groups=train["product_key"])
    rows.append((label, Xi.shape[1], rmse_log(oof_i, y)))

print(f"{'特徴量':<26}{'列数':>6}{'CV(GroupKFold)':>18}")
print("-" * 50)
for label, ncol, s in rows:
    print(f"{label:<24}{ncol:>6}{s:>18.4f}")

print("\n--- 読み方 ---")
print("  list_price だけでも 0.71 -> 0.19 と大きく改善する(定価は実売価格とほぼ比例するので当然)。")
print("  discount_rate だけだと 0.70 で、ほとんど効かない(割引率だけでは価格の水準が分からない)。")
print("  だが2つそろうと 0.068。掛け算すれば答えそのものだからだ。")
print("  → リークは『1列だけを見て』探すと見逃す。列の意味と、列どうしの関係を見る。")

# STEP 2: 検出方法1 — 目的変数との相関(単独で見る方法の限界)
print("\n--- 目的変数(log price)との相関 ---")
num_cols = [c for c in train.columns if pd.api.types.is_numeric_dtype(train[c]) and c != "price"]
for c in num_cols:
    r = float(np.corrcoef(train[c].to_numpy(dtype=float), y)[0, 1])
    flag = "  ← 怪しい" if abs(r) > 0.5 else ""
    print(f"  {c:<16}{r:>+9.4f}{flag}")
print("\n  discount_rate の相関は +0.04。相関だけを見ていたら絶対に見つからなかった。")
print("  だが test に無い列だという事実だけで、一発で分かる。だから検出方法3が本命。")

## ⑥ 書いてみる: リーク候補の列を自動で洗い出す

新しいデータを受け取った日に**必ず最初に回すチェック**として関数化しておく価値がある。
unit01 で書いた `sanity` dict の、リーク版だ。

次のセルで、2つの手掛かりを合わせた `suspicious` を作ろう。

**手掛かり1: `train` にあって `test` に無い列**(ただし目的変数 `price` は除く)

**手掛かり2: 目的変数との相関の絶対値が `0.5` より大きい数値列**
  - 対象は `train` の**数値列**(`pd.api.types.is_numeric_dtype` が True)から `price` を除いたもの
  - 相関は `np.corrcoef(train[c].to_numpy(dtype=float), y)[0, 1]`

作るもの:

- `suspicious` — 上の2つの**和集合**を、**アルファベット順に並べた Python のリスト**
  (`sorted(set(...) | set(...))` で作れる。`|` は集合の和 = C# の `Union`)
- `corr_list_price` — `list_price` と目的変数 `y` の相関係数(float)

5〜8行で書ける。`train` は元のまま(`days` 列を足した `train_f` ではない)を使うこと。

> **なぜ和集合なのか**: どちらか一方の手掛かりに引っかかれば「疑わしい」として人間の目に上げたいから。
> 自動判定で捨てるのではなく、**人間が『この列は予測時点で手に入るか』を判断する候補リスト**を作るのが目的だ。

In [ ]:
suspicious = None
corr_list_price = None
# ここに書く(ヒント: only_in_train = set(train.columns) - set(test.columns) - {"price"}
#           数値列は [c for c in train.columns if pd.api.types.is_numeric_dtype(train[c]) and c != "price"])

print("suspicious:", suspicious)
print("corr_list_price:", corr_list_price)

In [ ]:
# ===== チェックポイント 4: リーク候補の洗い出し =====
check("D-1 suspicious の件数",
      len(suspicious) if isinstance(suspicious, (list, tuple)) else None, 2,
      hint="test に無い列(price を除く)が1つ、相関 0.5 超えの列が1つ。重複はないので合計2つ。")

check("D-2 suspicious の中身(アルファベット順のリスト)",
      list(suspicious) if isinstance(suspicious, (list, tuple)) else None,
      ["discount_rate", "list_price"],
      hint="sorted(set(A) | set(B)) で作る。price が入っているなら除外を忘れている。"
           "record_id など文字列列が入っているなら数値列に絞れていない。")

check("D-3 corr_list_price", corr_list_price, 0.7549687245909541,
      hint="np.corrcoef(train['list_price'].to_numpy(dtype=float), y)[0, 1]。"
           "y は log1p 済みの価格。生の price との相関(0.98)を出していないか確認。")

check("D-4 suspicious は list か(set のままではないか)",
      1.0 if isinstance(suspicious, list) else 0.0, 1.0,
      hint="set のままだと順序が保証されない。sorted(...) はリストを返すのでそれを入れる。")

print("\n(4つとも [OK] になったら答え合わせへ)")
print("補足: この関数が挙げた2列を、最後は人間が判断する。")
print("      discount_rate → test に無い。使えない。除外。")
print("      list_price    → test にある。出品時点で分かる情報なので使ってよい。")
print("      『自動で捨てる』のではなく『自動で候補に挙げて人間が決める』のが実務の形。")

---
## 答え合わせ: 今日の全部を1枚に並べる

コンペ中は `test` の正解が見えない。今回は教材なので、**手元のCVがどれだけ本番を言い当てたか**を最後に全部並べる。

見るポイントは3つ。

1. **同じモデル・同じデータでも、分割方法だけで CV は倍近く動く。** どれを信じるかは自分で決めるしかない
2. **「CVが良い」ことに意味はない。「CVが本番を言い当てる」ことに意味がある。**
   0.6078 という美しい数字より、1.1119 という汚い数字の方がはるかに価値がある
3. **CV と LB がズレたときは、まず分割を疑う。** モデルではない

> ⑦ をまだ書いていない項目は「未記入」と表示される。書いてから戻ってきて再実行しよう。

In [ ]:
# GOAL: 今日測った全部を1枚の表にして、「どの検証を信じるべきだったか」を確認する

def fmt(v):
    return "  未記入" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:8.4f}"


print("=" * 74)
print("【A】ベースライン: model_code ごとの平均 log price(概念2)")
print("-" * 74)
print(f"{'検証方法':<38}{'RMSLE':>10}{'本番との差':>12}")
print(f"{'KFold(5, shuffle=True)':<36}{fmt(score_kf):>10}{score_kf - score_lb:>12.4f}   甘すぎ")
print(f"{'GroupKFold(4, groups=site)':<36}{fmt(globals().get('score_site_group')):>10}"
      f"{'':>12}   group キーの選択ミス")
print(f"{'GroupKFold(5, groups=product_key)':<36}{fmt(score_gkf):>10}{score_gkf - score_lb:>12.4f}   ほぼ的中")
print(f"{'本番LB':<36}{fmt(score_lb):>10}{0.0:>12.4f}")

print()
print("=" * 74)
print("【B】LightGBM(model_code を使わない特徴量)(概念3)")
print("-" * 74)
print(f"{'検証方法':<38}{'RMSLE':>10}{'本番との差':>12}")
print(f"{'KFold(5, shuffle=True)':<36}{fmt(score_random):>10}{score_random - score_lb_lgb:>12.4f}   やや甘い")
print(f"{'TimeSeriesSplit(5) ソート忘れ':<32}{fmt(score_unsorted):>10}{score_unsorted - score_lb_lgb:>12.4f}   ただのランダム分割")
print(f"{'TimeSeriesSplit(5) 正しく採点':<32}{fmt(score_right):>10}{score_right - score_lb_lgb:>12.4f}   やや辛い(安全側)")
print(f"{'TimeSeriesSplit(5) 未検証行を0で採点':<28}{fmt(score_wrong):>10}{'':>12}   バグ")
print(f"{'日付ホールドアウト(2026-02-01)':<31}{fmt(globals().get('score_time_holdout')):>10}"
      f"{'':>12}   1回きりなので揺れる")
print(f"{'本番LB':<36}{fmt(score_lb_lgb):>10}{0.0:>12.4f}")

print()
print("=" * 74)
print("【C】ターゲットリーク(GroupKFold 固定)(概念4)")
print("-" * 74)
print(f"{'リーク列なし':<36}{fmt(score_clean):>10}   本番に持っていける")
print(f"{'リーク列あり':<36}{fmt(score_leak):>10}   本番では再現しない(test に列が無い)")

print()
print("=" * 74)
print("結論:")
print("  1. 同じモデル・同じデータで、分割を変えるだけで CV は 0.52 〜 1.11 まで動いた。")
print("     『CV が良い』は何の情報でもない。『CV が本番を言い当てる』かどうかが全て。")
print("  2. 正しい分割は、データの生成過程から決まる。")
print("     同じ実体が複数行 -> GroupKFold。未来を当てる -> TimeSeriesSplit。両方 -> 両方。")
print("  3. 分割を正しくしても、特徴量にリークがあれば意味がない(【C】)。両方を毎回チェックする。")
print("=" * 74)

---
## 振り返り(自己評価 + TIL)

以下に**1〜2文ずつ**、自分の言葉で書いてみよう。書いた内容はセッション終了時の学習ノートと
スキルレベルの判定に使う(空欄でも先に進めるが、言語化すると定着が大きく変わる)。

**1. 今日学んだことを自分の言葉で:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック — 次の4つに、資料を見ずに答えられる?**

- 「OOF 予測とは何か」を、配列の長さと「各行が何回埋まるか」に触れながら説明できる?
- 「`GroupKFold` の group に何を選ぶか」の判断基準を1文で言える?
- 「`TimeSeriesSplit` を使うときに必ずやること」を2つ挙げられる?(1つは分割前、1つは採点時)
- 同僚が「CV が 0.05 まで下がりました!」と言ってきた。**最初に確認する3つ**は?

> (ここに書く)

---
## まとめ

### 今日学んだこと

| # | 概念 | 一言でいうと |
|---|---|---|
| 1 | ホールドアウトの限界 | 乱数の種を変えるだけでスコアが 0.05 動いた。**その上で 0.005 の改善は判断できない** |
| 2 | **OOF 予測** | 全行に「その行を学習に使っていないモデルの予測」を1つずつ入れた配列。長さは `len(train)` |
| 3 | OOF の作り方 | **(a)** `np.full(n, np.nan)` で箱 **(b)** fold ごとにモデルを作り直す **(c)** `oof[va_idx] = pred` で詰め戻す |
| 4 | `splitter.split()` | データではなく**行番号の組**を yield するイテレータ。実データを取るのは `X.iloc[idx]` |
| 5 | splitter の選択 | **データの生成過程**から決める。ランダム / 層化 / グループ / 時系列 |
| 6 | **グループリーク** | 同じ実体の行が train と valid に割れると、答えを見てから当てることになる |
| 7 | group キーの決め方 | **本番で未知になるもの**を group にする。`site` を選ぶとむしろ甘くなった(0.52) |
| 8 | 未知キー率 | 検証と本番で「初めて見るキーの割合」が揃っているか。**1つの数字で診断できる** |
| 9 | **時間リーク** | ランダム分割は「未来で学習して過去を当てる」。本番と向きが逆 |
| 10 | `TimeSeriesSplit` の罠1 | **日付列を見ない**。使う前に `sort_values` しないと、ただのランダム分割になる |
| 11 | `TimeSeriesSplit` の罠2 | **先頭ブロックは検証されない**。`np.zeros` 初期化 + 全行採点で 3.83 という出鱈目が出た |
| 12 | **ターゲットリーク** | 予測時点では手に入らない情報。CV 0.0681 = 良すぎる = 疑う |
| 13 | リークの検出3手法 | 相関 / 重要度の集中 / **その列は予測時点で手に入るか**(train と test の列の差分) |
| 14 | 組み合わせのリーク | `discount_rate` 単独の相関は +0.04。**相関だけ見ていると見逃す** |
| 15 | 実験の作法 | **一度に1つだけ変える**。分割を固定して特徴量を動かす、逆も同じ |
| 16 | 良い CV とは | 「値が小さい CV」ではなく「**本番を言い当てる CV**」 |

### この先どこで使うか(先読み)

- **unit03(GBDT)** — 今日の `run_lgb_cv` の形はそのまま使う。early stopping で `eval_set` に渡すのは
  「その fold の valid」であって全データではない、という話も今日の理解の上に乗る。
- **unit04(特徴量エンジニアリング)** — **target encoding** は「最も効くが最もリークしやすい」特徴量の代表。
  今日の OOF ループの中で統計を作る(**OOF target encoding**)ことで初めて安全になる。
  今日書いた `fit_predict_mean` は、実はその原型そのものだ。
  また `Pipeline` は「前処理を fit するのは学習 fold だけ」を**構造的に保証**する道具として出てくる。
- **unit06(名寄せ)** — 今日 `product_key` として与えられていた group キーを、**自分で作れるようになる**。
  実際のコンペでは親切な ID 列は無く、表記ゆれのある商品名から同一性を判定する必要がある。
- **unit10(キャップストーン)** — 今日作った OOF 配列を**横に並べた行列** `(n_train, n_models)` が、
  アンサンブル(ブレンド・スタッキング)の入力そのものになる。OOF が作れないと unit10 は始まらない。
- **実務** — 新しい予測タスクを受け取ったら、モデルを選ぶ前に **(1) 1行は何を表すか (2) 同じ実体の行はあるか
  (3) 本番では何が未知か (4) 本番で手に入らない列が混ざっていないか** の4つを固める。
  今日のレッスンはこの4つを数字で確かめる練習だった。

### 次にやること

**演習 `ex01_manual_oof` へ進もう。lesson.ipynb を見ながらで OK。**
思い出せない API があれば ② の表に戻ればいい。暗記ではなく、**どこを見れば分かるか**を覚えているのが実務の状態だ。

演習は4本:

| 演習 | 内容 |
|---|---|
| `ex01_manual_oof` | OOF 予測を作る関数を書く(今日の ⑦ の一般化) |
| `ex02_group_and_time_split` | GroupKFold / TimeSeriesSplit を使い分ける |
| `ex03_detect_duplicate_leak` | 重複・リーク列を検出する |
| `ex04_capstone` | リークのない分割を設計し、検証済み行を監査 |